# 02 — Agent Definition

Assembles the single LLM + four tools into a LangGraph ReAct agent. The LLM reasons between every tool call; nothing is hard coded into a fixed sequence.

**Tools:** `eligibility_screener`, `practice_matcher`, `payment_estimator`, `program_availability` — all four are bound in `agent/graph.py` `TOOLS`.

Out of scope handling is not a tool. The system prompt (PTCF form, written for internal advisors) makes the agent decline irrelevant input and redirect, and drives a short elicitation flow that gathers the client's profile (state, acreage, current practices, primary resource concern; county is optional since payment data is state level) across turns.

## Setup

In [18]:
import asyncio
import sys

if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

In [19]:
%load_ext autoreload
%autoreload 2

from nrcs_navigator import config
from nrcs_navigator.agent.graph import build_agent

# Prereqs: run notebook 01 first (payment_rates + eCFR vector store populated),
# the Postgres container is up, and OPENAI_API_KEY is set in .env -- both the
# premier model and the eligibility_screener query embedding use it.
print("imports ready")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
imports ready


## Build the agent

In [20]:
# Build the ReAct agent on the premier model. model_name comes from config/.env;
# build_agent(config.CHEAP_MODEL) would build the same agent on the cheaper leg.
# Only eligibility_screener is bound right now (see agent/graph.py TOOLS).
agent = build_agent(config.PREMIER_MODEL)
print(f"agent built on {config.PREMIER_MODEL}")

agent built on gpt-4o


## Run an in-scope query

A realistic advisor question with enough profile detail to proceed. The agent reasons (ReAct), calls the tools it needs, and answers with citations.

In [21]:
# A realistic in-scope question from an advisor about a client. The agent reasons
# (ReAct): it should call the tools it needs and answer with citations. thread_id
# keys this conversation in the Postgres checkpointer.
result = agent.invoke(
    {"messages": [("user",
        "I have a client with about 400 acres of cropland in Iowa. Their main "
        "resource concern is soil erosion. Which NRCS program fits, and what "
        "range of payments could they expect?")]},
    config={"configurable": {"thread_id": "demo-eqip"}},
)

# Full ReAct trace: the question, any tool calls + tool output, then the answer.
for message in result["messages"]:
    message.pretty_print()

Checking available programs...
Matching practices...
================================ Human Message =================================

I have a client with about 400 acres of cropland in Iowa. Their main resource concern is soil erosion. Which NRCS program fits, and what range of payments could they expect?
================================== Ai Message ==================================
Tool Calls:
  eligibility_screener (call_x61FvtUKFn4qTfBzhyN5MSgk)
 Call ID: call_x61FvtUKFn4qTfBzhyN5MSgk
  Args:
    query: 400 acres of cropland in Iowa with a primary resource concern of soil erosion
  practice_matcher (call_bc4CNf1kGTj0BCafoNrObZzK)
 Call ID: call_bc4CNf1kGTj0BCafoNrObZzK
  Args:
  program_availability (call_k46aOFRjY5ITbGsLJk9xE2Dd)
 Call ID: call_k46aOFRjY5ITbGsLJk9xE2Dd
  Args:
    state: Iowa
================================= Tool Message =================================
Name: eligibility_screener

[7 CFR 1468.30] § 1468.30 Program requirements. (ACEP)
§ 1468.30 Program requir

## Demonstrate graceful rejection

In [22]:
# An out-of-scope request: CRP is administered by FSA, not NRCS. The scope guard
# in the system prompt should make the agent decline and redirect to the local
# FSA office WITHOUT calling any tool.
result = agent.invoke(
    {"messages": [("user",
        "Can you help my client enroll in the Conservation Reserve Program (CRP)?")]},
    config={"configurable": {"thread_id": "demo-crp"}},
)
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The Conservation Reserve Program (CRP) is administered by the Farm Service Agency (FSA), not the NRCS. Your client should contact their local FSA office for assistance with CRP enrollment.
